# Extract Semantic Model Metadata
Extract granular metadata from Semantic Models using **semantic-link-labs** and write it to **managed Delta tables**.

Primary output tables:
- `lineage_semantic_models`
- `lineage_semantic_model_tables`
- `lineage_semantic_model_measures`
- `lineage_semantic_model_columns`
- `lineage_semantic_model_relationships`
- `lineage_semantic_model_functions`
- `lineage_semantic_model_dependencies`
- `lineage_semantic_model_extraction_log`

## 1. Imports

**Note**: Packages loaded from Fabric Environment (no installation needed).

In [78]:
# All packages available from attached Fabric Environment
import json
from datetime import datetime
from typing import Any, Dict, List

from pyspark.sql import SparkSession  # type: ignore[reportMissingImports]
from sempy.fabric import list_datasets, get_model_calc_dependencies  # type: ignore[reportMissingImports]
from sempy_labs.tom import connect_semantic_model  # type: ignore[reportMissingImports]

spark = globals().get("spark")
if spark is None:
    spark = SparkSession.builder.getOrCreate()

print("✅ Imports successful (semantic-link-labs + spark)")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 95, Finished, Available, Finished, False)

✅ Imports successful (semantic-link-labs + spark)


## 2. Configuration
Parameters passed from custom item via Spark Livy API.

In [79]:
# These will be provided as parameters when triggered via Livy
# For testing, set manually:
WORKSPACE_IDS = ["2b1d454b-61a1-4abb-96c3-2b476d945a96", "242dd37c-864e-4fec-af02-1caaf4d3b1a2" ]  # Can extract from multiple workspaces
LAKEHOUSE_ID = "8f644af3-54bc-4525-8cf8-7b78e5b01cd5"

print(f"Extracting from {len(WORKSPACE_IDS)} workspace(s)")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 96, Finished, Available, Finished, False)

Extracting from 2 workspace(s)


## 3. Extract Semantic Models

In [80]:
# Helpers: serialization, row shaping, and Delta table I/O
def _serialize_value(value: Any) -> Any:
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, datetime):
        return value.isoformat()
    return json.dumps(value, default=str)


def _sanitize_column_name(name: str) -> str:
    sanitized_chars = []
    for ch in str(name).strip():
        if ch.isalnum() or ch == "_":
            sanitized_chars.append(ch.lower())
        else:
            sanitized_chars.append("_")

    sanitized = "".join(sanitized_chars)
    while "__" in sanitized:
        sanitized = sanitized.replace("__", "_")
    sanitized = sanitized.strip("_")

    if not sanitized:
        sanitized = "col"
    if sanitized[0].isdigit():
        sanitized = f"col_{sanitized}"

    return sanitized


def _add_sanitized_row_fields(prepared_row: Dict[str, Any], raw_row: Dict[str, Any]) -> None:
    used_names = set(prepared_row.keys())

    for raw_key, raw_value in raw_row.items():
        base_name = _sanitize_column_name(raw_key)
        final_name = base_name
        suffix = 2

        while final_name in used_names:
            final_name = f"{base_name}_{suffix}"
            suffix += 1

        prepared_row[final_name] = _serialize_value(raw_value)
        used_names.add(final_name)


def _prepare_rows(
    rows: List[Dict[str, Any]],
    workspace_id: str,
    model_id: str,
    model_name: str,
    extraction_timestamp: str
) -> List[Dict[str, Any]]:
    prepared_rows = []

    for row in rows:
        prepared_row = {
            "workspace_id": workspace_id,
            "model_id": model_id,
            "model_name": model_name,
            "extraction_timestamp": extraction_timestamp
        }
        _add_sanitized_row_fields(prepared_row, row)
        prepared_rows.append(prepared_row)

    return prepared_rows


def _delta_table_exists(table_name: str) -> bool:
    return bool(spark.catalog.tableExists(table_name))


def _load_delta_table(table_name: str):
    return spark.table(table_name)


def _drop_table_if_exists(table_name: str) -> None:
    if _delta_table_exists(table_name):
        spark.sql(f"DROP TABLE IF EXISTS `{table_name}`")


def _prepare_table_for_write(table_name: str, mode: str) -> None:
    if mode == "overwrite":
        _drop_table_if_exists(table_name)


def _append_to_delta(table_name: str, rows: List[Dict[str, Any]], mode: str = "append") -> int:
    if not rows:
        return 0

    _prepare_table_for_write(table_name, mode)
    df = spark.createDataFrame(rows)

    writer = (
        df.write
        .format("delta")
        .option("mergeSchema", "true")
    )

    if mode == "overwrite":
        writer.mode("overwrite").saveAsTable(table_name)
    else:
        writer.mode("append").saveAsTable(table_name)

    return len(rows)


def _build_model_row(
    model_id: str,
    workspace_id: str,
    model_name: str,
    extraction_timestamp: str,
    duration: float,
    model_info_row: Dict[str, Any],
    counts: Dict[str, int]
) -> Dict[str, Any]:
    row = {
        "workspace_id": workspace_id,
        "model_id": model_id,
        "model_name": model_name,
        "extraction_timestamp": extraction_timestamp,
        "extraction_method": "sempy_labs.tom.connect_semantic_model",
        "extraction_duration": duration,
        "raw_model_info": json.dumps(model_info_row, default=str)
    }

    _add_sanitized_row_fields(row, model_info_row)

    for key, value in counts.items():
        row[f"count_{key}"] = value

    return row


def _build_log_rows(
    workspace_id: str,
    model_id: str,
    model_name: str,
    extraction_timestamp: str,
    status: str,
    error_message: str | None = None
) -> List[Dict[str, Any]]:
    if not error_message:
        return []

    return [{
        "workspace_id": workspace_id,
        "model_id": model_id,
        "model_name": model_name,
        "extraction_timestamp": extraction_timestamp,
        "status": status,
        "message": error_message
    }]

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 48, Finished, Available, Finished, False)

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 97, Finished, Available, Finished, False)

In [81]:
# Extract semantic model functions
def _extract_functions(tom) -> List[Dict[str, Any]]:
    functions = []
    for function in tom.all_functions():
        functions.append({
            "Name": function.Name,
            "Expression": function.Expression,
            "Description": function.Description if hasattr(function, "Description") else None
        })
    return functions

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 98, Finished, Available, Finished, False)

In [102]:
# Extract semantic model dependencies using semantic link labs
def _extract_dependencies(model_name: str, workspace_id: str, model_id: str = None) -> List[Dict[str, Any]]:
    """Extract all dependencies between measures, columns, and tables using ModelCalcDependencies."""
    dependencies = []
    
    try:
        # Get the ModelCalcDependencies object - it's a context manager that yields ModelCalcDependencies
        print(f"  └─ Extracting dependencies for '{model_name}'...")
        
        with get_model_calc_dependencies(model_name, workspace=workspace_id) as calc_deps:
            # calc_deps is a ModelCalcDependencies object, not an iterator
            # Access the dependencies DataFrame
            deps_df = calc_deps.dependencies_df
            
            if deps_df is None or len(deps_df) == 0:
                print(f"  ⚠️  No dependencies found (model may have no calculated dependencies)")
                return dependencies
            
            print(f"  └─ Found {len(deps_df)} dependency records")
            
            # Build lineage tag lookup map from a separate TOM connection
            lineage_tag_map = {}
            if model_id:
                try:
                    print(f"  └─ Building lineage tag map...")
                    with connect_semantic_model(dataset=model_id, workspace=workspace_id, readonly=True) as tom:
                        for col in tom.all_columns():
                            if hasattr(col, "LineageTag") and col.LineageTag:
                                key = f"'{col.Parent.Name}'[{col.Name}]"
                                lineage_tag_map[key] = col.LineageTag
                        for measure in tom.all_measures():
                            if hasattr(measure, "LineageTag") and measure.LineageTag:
                                key = f"'{measure.Parent.Name}'[{measure.Name}]"
                                lineage_tag_map[key] = measure.LineageTag
                    print(f"  └─ Built {len(lineage_tag_map)} lineage tag mappings")
                except Exception as e:
                    print(f"  ⚠️  Warning: Could not build lineage tag map: {str(e)}")
            
            # Convert DataFrame rows to dependency records with lineage tags
            for _, row in deps_df.iterrows():
                full_obj_name = row.get("Full Object Name", "")
                ref_full_obj_name = row.get("Referenced Full Object Name", "")
                
                dep_record = {
                    "ObjectName": row.get("Object Name", ""),
                    "ObjectType": row.get("Object Type", ""),
                    "TableName": row.get("Table Name", ""),
                    "ObjectLineageTag": lineage_tag_map.get(full_obj_name, ""),
                    "ReferencedObjectName": row.get("Referenced Object", ""),
                    "ReferencedObjectType": row.get("Referenced Object Type", ""),
                    "ReferencedTableName": row.get("Referenced Table", ""),
                    "ReferencedObjectLineageTag": lineage_tag_map.get(ref_full_obj_name, ""),
                    "ReferencedExpression": row.get("Expression", ""),
                    "FullObjectName": full_obj_name,
                    "ReferencedFullObjectName": ref_full_obj_name,
                    "DependencyType": "Direct"
                }
                dependencies.append(dep_record)
            
    except Exception as e:
        print(f"  ⚠️  Error extracting dependencies: {type(e).__name__}: {str(e)}")
    
    return dependencies

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 124, Finished, Available, Finished, False)

In [83]:
# Extract semantic model relationships
def _extract_relationships(tom) -> List[Dict[str, Any]]:
    relationships = []
    for rel in tom.model.Model.Relationships:
        relationships.append({
            "Name": rel.Name if hasattr(rel, "Name") else f"{rel.FromTable.Name}_{rel.ToTable.Name}",
            "FromTable": rel.FromTable.LineageTag,
            "FromColumn": rel.FromColumn.LineageTag,
            "ToTable": rel.ToTable.LineageTag,
            "ToColumn": rel.ToColumn.LineageTag,
            "IsActive": rel.IsActive,
            "CrossFilteringBehavior": str(rel.CrossFilteringBehavior),
            "FromCardinality": str(rel.FromCardinality) if hasattr(rel, "FromCardinality") else None,
            "ToCardinality": str(rel.ToCardinality) if hasattr(rel, "ToCardinality") else None
        })
    return relationships

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 100, Finished, Available, Finished, False)

In [84]:
# Extract semantic model columns
def _extract_columns(tom) -> List[Dict[str, Any]]:
    columns = []
    for column in tom.all_columns():
        columns.append({
            "Name": column.Name,
            "DataType": str(column.DataType),
            "Description": column.Description if hasattr(column, "Description") else None,
            "Table": column.Parent.Name,
            "SourceColumn": column.SourceColumn if hasattr(column, "SourceColumn") else None,
            "LineageTag": column.LineageTag if hasattr(column, "LineageTag") else None,
            "SourceLineageTag": column.SourceLineageTag if hasattr(column, "SourceLineageTag") else None,
            "IsHidden": column.IsHidden if hasattr(column, "IsHidden") else None,
            "IsKey": column.IsKey if hasattr(column, "IsKey") else None,
            "Expression": column.Expression if hasattr(column, "Expression") else None

        })
    return columns

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 101, Finished, Available, Finished, False)

In [85]:
# Extract semantic model measures
def _extract_measures(tom) -> List[Dict[str, Any]]:
    measures = []
    for measure in tom.all_measures():
        measures.append({
            "LineageTag": measure.LineageTag if hasattr(measure, "LineageTag") else None,
            "Name": measure.Name,
            "Expression": measure.Expression,
            "Description": measure.Description if hasattr(measure, "Description") else None,
            "Table": measure.Parent.Name,
            "FormatString": measure.FormatString if hasattr(measure, "FormatString") else None
        })
    return measures

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 102, Finished, Available, Finished, False)

In [86]:
# Extract semantic model tables
def _extract_tables(tom) -> List[Dict[str, Any]]:
    tables = []

    if hasattr(tom, "all_tables"):
        for table in tom.all_tables():
            tables.append({
                "Name": table.Name,
                "Description": table.Description if hasattr(table, "Description") else None,
                "IsHidden": table.IsHidden if hasattr(table, "IsHidden") else None
            })
        return tables

    model_tables = getattr(getattr(getattr(tom, "model", None), "Model", None), "Tables", None)
    if model_tables is not None:
        for table in model_tables:
            tables.append({
                "Name": table.Name if hasattr(table, "Name") else str(table),
                "Description": table.Description if hasattr(table, "Description") else None,
                "IsHidden": table.IsHidden if hasattr(table, "IsHidden") else None
            })
        return tables

    # Last-resort fallback if direct table APIs are unavailable.
    inferred_names = set()
    for column in tom.all_columns():
        if hasattr(column, "Parent") and hasattr(column.Parent, "Name"):
            inferred_names.add(column.Parent.Name)
    for measure in tom.all_measures():
        if hasattr(measure, "Parent") and hasattr(measure.Parent, "Name"):
            inferred_names.add(measure.Parent.Name)

    for name in sorted(inferred_names):
        tables.append({
            "Name": name,
            "Description": None,
            "IsHidden": None
        })

    return tables

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 103, Finished, Available, Finished, False)

In [103]:
# Extract semantic model: orchestration and shaping for Delta output tables
def extract_semantic_model(model_id: str, workspace_id: str, model_info_row: Dict[str, Any] | None = None) -> dict:
    """Extract semantic model metadata and return rows grouped by target Delta table."""
    print(f"\n📊 Extracting Semantic Model: {model_id}")

    extraction_start = datetime.now()
    model_info_row = model_info_row or {}

    try:
        print("  └─ Connecting to semantic model...")
        with connect_semantic_model(dataset=model_id, workspace=workspace_id, readonly=True) as tom:
            tables = _extract_tables(tom)
            measures = _extract_measures(tom)
            columns = _extract_columns(tom)
            relationships = _extract_relationships(tom)
            functions = _extract_functions(tom)
        
        # Determine model name for dependencies API (requires name, not ID)
        model_name = (
            model_info_row.get("Dataset Name")
            or model_info_row.get("Name")
            or model_info_row.get("name")
            or str(model_id)
        )
        
        # Extract dependencies outside the TOM context (uses separate connection with model name)
        dependencies = _extract_dependencies(model_name, workspace_id, model_id)

        extraction_end = datetime.now()
        extraction_timestamp = extraction_end.isoformat()
        duration = (extraction_end - extraction_start).total_seconds()

        counts = {
            "tables": len(tables),
            "measures": len(measures),
            "columns": len(columns),
            "relationships": len(relationships),
            "functions": len(functions),
            "dependencies": len(dependencies)
        }

        print(
            f"  ✅ Extracted: {counts['tables']} tables, {counts['measures']} measures, "
            f"{counts['columns']} columns, {counts['relationships']} relationships, {counts['dependencies']} dependencies"
        )

        table_rows = {
            "lineage_semantic_models": [_build_model_row(
                model_id=model_id,
                workspace_id=workspace_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp,
                duration=duration,
                model_info_row=model_info_row,
                counts=counts
            )],
            "lineage_semantic_model_tables": _prepare_rows(
                rows=tables,
                workspace_id=workspace_id,
                model_id=model_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp
            ),
            "lineage_semantic_model_measures": _prepare_rows(
                rows=measures,
                workspace_id=workspace_id,
                model_id=model_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp
            ),
            "lineage_semantic_model_columns": _prepare_rows(
                rows=columns,
                workspace_id=workspace_id,
                model_id=model_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp
            ),
            "lineage_semantic_model_relationships": _prepare_rows(
                rows=relationships,
                workspace_id=workspace_id,
                model_id=model_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp
            ),
            "lineage_semantic_model_functions": _prepare_rows(
                rows=functions,
                workspace_id=workspace_id,
                model_id=model_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp
            ),
            "lineage_semantic_model_dependencies": _prepare_rows(
                rows=dependencies,
                workspace_id=workspace_id,
                model_id=model_id,
                model_name=model_name,
                extraction_timestamp=extraction_timestamp
            )
        }

        return {
            "artifactId": model_id,
            "artifactType": "SemanticModel",
            "workspaceId": workspace_id,
            "timestamp": extraction_timestamp,
            "modelName": model_name,
            "status": "success",
            "counts": counts,
            "tableRows": table_rows,
            "metadata": {
                "extractionDuration": duration,
                "status": "success"
            }
        }

    except Exception as e:
        extraction_end = datetime.now()
        extraction_timestamp = extraction_end.isoformat()
        duration = (extraction_end - extraction_start).total_seconds()
        model_name = (
            model_info_row.get("Dataset Name")
            or model_info_row.get("Name")
            or model_info_row.get("name")
            or str(model_id)
        )

        error_message = str(e)
        print(f"  ❌ Error: {error_message}")

        return {
            "artifactId": model_id,
            "artifactType": "SemanticModel",
            "workspaceId": workspace_id,
            "timestamp": extraction_timestamp,
            "modelName": model_name,
            "status": "error",
            "counts": {},
            "tableRows": {
                "lineage_semantic_model_extraction_log": _build_log_rows(
                    workspace_id=workspace_id,
                    model_id=model_id,
                    model_name=model_name,
                    extraction_timestamp=extraction_timestamp,
                    status="error",
                    error_message=error_message
                )
            },
            "metadata": {
                "status": "error",
                "errorMessage": error_message,
                "extractionDuration": duration
            }
        }

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 125, Finished, Available, Finished, False)

## 4. Process All Workspaces

In [105]:
# Get all Semantic Models from target workspaces and prepare rows per output Delta table
all_results = []
table_write_counts: Dict[str, int] = {}

EXPECTED_TABLES = [
    "lineage_semantic_models",
    "lineage_semantic_model_tables",
    "lineage_semantic_model_measures",
    "lineage_semantic_model_columns",
    "lineage_semantic_model_relationships",
    "lineage_semantic_model_functions",
    "lineage_semantic_model_dependencies",
    "lineage_semantic_model_extraction_log"
 ]

WRITE_MODE = "append"  # set to "overwrite" for clean reruns

table_row_batches: Dict[str, List[Dict[str, Any]]] = {table_name: [] for table_name in EXPECTED_TABLES}

for workspace_id in WORKSPACE_IDS:
    print(f"\n🔍 Processing workspace: {workspace_id}")

    try:
        datasets_df = list_datasets(workspace=workspace_id)

        if datasets_df.empty:
            print("  No Semantic Models found in workspace")
            continue

        models = datasets_df.to_dict("records")
        print(f"  Found {len(models)} Semantic Model(s)")

        for model in models:
            model_id = model.get("Dataset ID") or model.get("id")
            model_name = model.get("Dataset Name") or model.get("name") or model_id

            if not model_id:
                print(f"  ⚠️  Skipping model '{model_name}' - no valid ID")
                continue

            print(f"  Processing: {model_name}")
            result = extract_semantic_model(
                model_id=model_id,
                workspace_id=workspace_id,
                model_info_row=model
            )
            all_results.append(result)

            for table_name, rows in result.get("tableRows", {}).items():
                if table_name not in table_row_batches:
                    table_row_batches[table_name] = []
                table_row_batches[table_name].extend(rows)

    except Exception as e:
        print(f"  ❌ Workspace error: {str(e)}")

print(f"\n✅ Extraction complete: {len(all_results)} Semantic Model(s) processed")
print("\nPrepared rows by output table:")
for table_name in sorted(table_row_batches.keys()):
    print(f"   {table_name}: {len(table_row_batches[table_name])}")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 127, Finished, Available, Finished, False)


🔍 Processing workspace: 2b1d454b-61a1-4abb-96c3-2b476d945a96
  Found 3 Semantic Model(s)
  Processing: Training Datamodel Solution

📊 Extracting Semantic Model: 80490e11-fa1f-4d7c-a5e3-fefa2d877221
  └─ Connecting to semantic model...
  └─ Extracting dependencies for 'Training Datamodel Solution'...
  └─ Found 85 dependency records
  └─ Building lineage tag map...
  └─ Built 47 lineage tag mappings
  ✅ Extracted: 7 tables, 6 measures, 41 columns, 7 relationships, 85 dependencies
  Processing: Almost empty

📊 Extracting Semantic Model: 90ce1fe5-5ea0-41d1-acc0-53d307f5f846
  └─ Connecting to semantic model...
  └─ Extracting dependencies for 'Almost empty'...
  └─ Found 1 dependency records
  └─ Building lineage tag map...
  └─ Built 2 lineage tag mappings
  ✅ Extracted: 1 tables, 1 measures, 1 columns, 0 relationships, 1 dependencies
  Processing: Human Resources Sample PBIX

📊 Extracting Semantic Model: 623a4171-5730-4246-b21d-c5538b642a26
  └─ Connecting to semantic model...
  └─ Ext

In [89]:
# Write: lineage_semantic_models
table_name = "lineage_semantic_models"
written_count = _append_to_delta(table_name, table_row_batches.get(table_name, []), mode=WRITE_MODE)
table_write_counts[table_name] = written_count
print(f"📝 {table_name}: wrote {written_count} row(s)")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 106, Finished, Available, Finished, False)

📝 lineage_semantic_models: wrote 6 row(s)


In [90]:
# Write: lineage_semantic_model_tables
table_name = "lineage_semantic_model_tables"
written_count = _append_to_delta(table_name, table_row_batches.get(table_name, []), mode=WRITE_MODE)
table_write_counts[table_name] = written_count
print(f"📝 {table_name}: wrote {written_count} row(s)")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 107, Finished, Available, Finished, False)

📝 lineage_semantic_model_tables: wrote 49 row(s)


In [91]:
# Write: lineage_semantic_model_measures
table_name = "lineage_semantic_model_measures"
written_count = _append_to_delta(table_name, table_row_batches.get(table_name, []), mode=WRITE_MODE)
table_write_counts[table_name] = written_count
print(f"📝 {table_name}: wrote {written_count} row(s)")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 108, Finished, Available, Finished, False)

📝 lineage_semantic_model_measures: wrote 46 row(s)


In [92]:
# Write: lineage_semantic_model_columns
table_name = "lineage_semantic_model_columns"
written_count = _append_to_delta(table_name, table_row_batches.get(table_name, []), mode=WRITE_MODE)
table_write_counts[table_name] = written_count
print(f"📝 {table_name}: wrote {written_count} row(s)")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 109, Finished, Available, Finished, False)

📝 lineage_semantic_model_columns: wrote 337 row(s)


In [93]:
# Write: lineage_semantic_model_relationships
table_name = "lineage_semantic_model_relationships"
written_count = _append_to_delta(table_name, table_row_batches.get(table_name, []), mode=WRITE_MODE)
table_write_counts[table_name] = written_count
print(f"📝 {table_name}: wrote {written_count} row(s)")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 110, Finished, Available, Finished, False)

📝 lineage_semantic_model_relationships: wrote 33 row(s)


In [94]:
# Write: lineage_semantic_model_functions
table_name = "lineage_semantic_model_functions"
written_count = _append_to_delta(table_name, table_row_batches.get(table_name, []), mode=WRITE_MODE)
table_write_counts[table_name] = written_count
print(f"📝 {table_name}: wrote {written_count} row(s)")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 111, Finished, Available, Finished, False)

📝 lineage_semantic_model_functions: wrote 0 row(s)


In [95]:
# Write: lineage_semantic_model_extraction_log
table_name = "lineage_semantic_model_extraction_log"
written_count = _append_to_delta(table_name, table_row_batches.get(table_name, []), mode=WRITE_MODE)
table_write_counts[table_name] = written_count
print(f"📝 {table_name}: wrote {written_count} row(s)")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 112, Finished, Available, Finished, False)

📝 lineage_semantic_model_extraction_log: wrote 0 row(s)


In [106]:
# Write: lineage_semantic_model_dependencies
table_name = "lineage_semantic_model_dependencies"
written_count = _append_to_delta(table_name, table_row_batches.get(table_name, []), mode=WRITE_MODE)
table_write_counts[table_name] = written_count
print(f"📝 {table_name}: wrote {written_count} row(s)")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 128, Finished, Available, Finished, False)

📝 lineage_semantic_model_dependencies: wrote 950 row(s)


## 5. Summary

In [107]:
# Display summary
success_count = sum(1 for r in all_results if r.get("status") == "success")
error_count = len(all_results) - success_count

print("\n" + "="*50)
print("EXTRACTION SUMMARY (SEMANTIC MODELS + DELTA)")
print("="*50)
print(f"Total Semantic Models: {len(all_results)}")
print(f"✅ Successful: {success_count}")
print(f"❌ Errors: {error_count}")
print("\nDelta tables written:")
for table_name in sorted(table_write_counts.keys()):
    print(f"- {table_name}: {table_write_counts[table_name]} row(s)")
print("="*50)

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 129, Finished, Available, Finished, False)


EXTRACTION SUMMARY (SEMANTIC MODELS + DELTA)
Total Semantic Models: 6
✅ Successful: 6
❌ Errors: 0

Delta tables written:
- lineage_semantic_model_dependencies: 950 row(s)


## 6. Delta Table Schemas

In [ ]:
# Display schema for each Delta table
table_names = [
    "lineage_semantic_models",
    "lineage_semantic_model_tables",
    "lineage_semantic_model_measures",
    "lineage_semantic_model_columns",
    "lineage_semantic_model_relationships",
    "lineage_semantic_model_functions",
    "lineage_semantic_model_dependencies",
    "lineage_semantic_model_extraction_log"
]

print("\n" + "="*80)
print("DELTA TABLE SCHEMAS")
print("="*80)

for table_name in table_names:
    try:
        if spark.catalog.tableExists(table_name):
            print(f"\n📋 {table_name}")
            print("-" * 80)
            
            # Get the table and display schema
            df = spark.table(table_name)
            
            # Display schema in a readable format
            for field in df.schema.fields:
                field_type = str(field.dataType)
                nullable = "nullable" if field.nullable else "not null"
                print(f"  {field.name:<40} {field_type:<30} {nullable}")
            
            print(f"\n  Total columns: {len(df.schema.fields)}")
        else:
            print(f"\n⚠️  {table_name}: Table does not exist")
    except Exception as e:
        print(f"\n⚠️  {table_name}: Error reading schema - {str(e)}")

print("\n" + "="*80)

StatementMeta(, 4703b0fe-64c0-4bb8-96a2-5922b49a69cc, 8, Finished, Available, Finished, False)


DELTA TABLE SCHEMAS


NameError: name 'EXPECTED_TABLES' is not defined

---
# 🧪 Testing Guide

## How to Test Semantic Model Extraction

### Option 1: Test Single Model (Recommended)
Use the test cell below to extract ONE model before running full workspace extraction.

### Option 2: Test Full Extraction
Update configuration (cell 4) and run all cells for complete workspace extraction.

## 🧪 Test: Extract Single Model

## 🧪 Verify Saved Data

In [99]:
# Inspect the Delta tables written by this notebook
table_names = [
    "lineage_semantic_models",
    "lineage_semantic_model_tables",
    "lineage_semantic_model_measures",
    "lineage_semantic_model_columns",
    "lineage_semantic_model_relationships",
    "lineage_semantic_model_functions",
    "lineage_semantic_model_dependencies",
    "lineage_semantic_model_extraction_log"
 ]

for table_name in table_names:
    try:
        if not _delta_table_exists(table_name):
            print(f"⚠️  {table_name}: table does not exist yet")
            continue

        row_count = _load_delta_table(table_name).count()
        print(f"📊 {table_name}: {row_count} row(s)")
        if row_count > 0:
            display(_load_delta_table(table_name).limit(5))
    except Exception as e:
        print(f"⚠️  {table_name}: not available yet ({e})")

StatementMeta(, 7b052f79-2324-4070-8c7e-91f84d566cf8, 116, Finished, Available, Finished, False)

📊 lineage_semantic_models: 39 row(s)


SynapseWidget(Synapse.DataFrame, b09aa4f9-66d9-400a-a02e-4a8cc330c5c6)

📊 lineage_semantic_model_tables: 245 row(s)


SynapseWidget(Synapse.DataFrame, adf60b34-ee83-45f5-8da4-78f2916a9561)

...

SynapseWidget(Synapse.DataFrame, 0757f57c-fa4e-4bc7-893c-fb1f6faffcc0)

⚠️  lineage_semantic_model_functions: table does not exist yet
📊 lineage_semantic_model_dependencies: 950 row(s)


SynapseWidget(Synapse.DataFrame, 9d9af4af-61d1-48b8-88bc-4718225327e7)

📊 lineage_semantic_model_extraction_log: 6 row(s)


SynapseWidget(Synapse.DataFrame, 981a402c-5643-4826-b46c-bdea1c9b05d7)

---
## 📋 Troubleshooting

### Common Issues:

**1. "ModuleNotFoundError: No module named 'sempy'"**
- Attach Fabric Environment with semantic-link packages (top toolbar → Environment dropdown)
- Verify environment is published (not Draft status)
- Run `00_setup.ipynb` to verify environment configuration

**2. "401 Unauthorized" or "403 Forbidden"**
- Verify workspace permissions (Contributor or higher required)
- Check model permissions (viewer access minimum)
- Ensure correct workspace/model IDs

**3. "Empty DataFrames"**
- Model might have no measures/columns (unlikely)
- Check if semantic-link-labs supports the model version
- Try different model

**4. "Timeout" or "Performance Issues"**
- Large models (1000+ measures) take 30-60 seconds
- Consider extracting fewer workspaces at once
- Use cell 11 to test single model first

### Expected Performance:
- Small model (< 50 measures): 5-10 seconds
- Medium model (50-500 measures): 10-30 seconds
- Large model (> 500 measures): 30-60 seconds

### Next Steps After Testing:
1. ✅ Verify extraction works with test model
2. Update `WORKSPACE_IDS` in cell 4 with target workspaces
3. Run full extraction (cells 4-8)
4. Check lakehouse for results
5. Proceed to `02_extract_reports.ipynb`
